# Thermal Nasal Respiration Analysis

Contactless respiratory monitoring from thermal infrared video — no sensors, no contact.

A YOLO-based detector localises the nostril region in each frame and extracts the mean thermal intensity as a respiration proxy signal.

---

## Environment Setup

Run this cell first. On **Google Colab** it clones the repo and installs dependencies. Locally it does nothing.


In [ ]:
import os, subprocess

def is_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False

if is_colab():
    subprocess.run(["git", "clone", "https://github.com/parvthacker/nasal-resp-demo.git"])
    os.chdir("/content/nasal-resp-demo")
    subprocess.run(["pip", "install", "-q", "ultralytics"])
    print("✅ Ready")
else:
    print("Local — skipping")

## Imports


In [ ]:
import cv2 as cv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
from scipy.signal import butter,filtfilt,savgol_filter
import time
from ultralytics import YOLO
import torch
from tqdm.notebook import tqdm

---

# 1. Configuration

Set the model path, data root, and participant IDs here before running anything else.

| Variable | Description |
|---|---|
| `model` | YOLO detector — localises face and nostril bounding boxes |
| `data_path` | Root directory containing participant folders |
| `pids` | List of participant IDs to process |


In [ ]:
model = YOLO('./models/best.pt',task='detect').to('cuda')
data_path = "."
pids = ["P01_1"]

In [ ]:
model.names

---

# 2. Nasal Localizer

Runs the YOLO detector on every frame to:

1. Detect the **face** and **nostril** bounding boxes
2. Compute a **containment metric** — checks that the nostril box sits correctly inside the face box (data quality flag)
3. Track **nostril visibility** frame-by-frame as a binary signal
4. Write an **annotated video** with a scrolling visibility timeline strip

**Outputs per participant:**
```
P01_1/
├── nasal-resp/           # raw mean intensity .npy arrays
├── nostril-visible/      # binary visibility .npy arrays
└── annotated_video/      # annotated .avi with visibility strip
```

In [ ]:
import os
import cv2 as cv
import numpy as np
import time
from tqdm import tqdm
import csv

ANNOTATED_DIR_NAME = "annotated_video"

STRIP_HEIGHT = 80

FACE_CLASS = 0
NOSTRIL_CLASS = 0

csv_path = os.path.join(data_path, "nostril_face_metricsws.csv")

# -------- INIT CSV --------
if not os.path.exists(csv_path):
    with open(csv_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            "pid", "video",
            "mean_containment",
            "containment_pass_frac",
            "valid_frac",
            "total_frames"
        ])

# -------- HELPER --------
def containment_ratio(inner, outer):
    xi1, yi1, xi2, yi2 = inner
    xo1, yo1, xo2, yo2 = outer

    inter_x1 = max(xi1, xo1)
    inter_y1 = max(yi1, yo1)
    inter_x2 = min(xi2, xo2)
    inter_y2 = min(yi2, yo2)

    inter_area = max(0, inter_x2 - inter_x1) * max(0, inter_y2 - inter_y1)
    inner_area = (xi2 - xi1) * (yi2 - yi1) + 1e-6

    return inter_area / inner_area


# -------- MAIN LOOP --------
for curr_pid in tqdm(pids, position=0):

    video_store_path = os.path.join(data_path, curr_pid)
    resp_store_path = os.path.join(data_path, curr_pid, "nasal-resp")
    nostril_visible_store_path = os.path.join(data_path, curr_pid, "nostril-visible")
    annotated_store_path = os.path.join(data_path, curr_pid, ANNOTATED_DIR_NAME)

    os.makedirs(resp_store_path, exist_ok=True)
    os.makedirs(nostril_visible_store_path, exist_ok=True)
    os.makedirs(annotated_store_path, exist_ok=True)

    for vid_names in tqdm(np.sort(os.listdir(video_store_path)),
                         position=1, leave=False,
                         desc=f"Processing {curr_pid}"):

        if not vid_names.endswith(".avi"):
            continue

        video_to_read = os.path.join(video_store_path, vid_names)
        cap = cv.VideoCapture(video_to_read)

        if not cap.isOpened():
            print(f"Failed to open: {video_to_read}")
            continue

        total_frames = int(cap.get(cv.CAP_PROP_FRAME_COUNT))  # may be wrong for truncated files
        fps = cap.get(cv.CAP_PROP_FPS)
        width = int(cap.get(cv.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv.CAP_PROP_FRAME_HEIGHT))

        out_h = height + STRIP_HEIGHT

        output_video_path = os.path.join(
            annotated_store_path,
            vid_names.replace(".avi", "_annotated.avi")
        )

        fourcc = cv.VideoWriter_fourcc(*'MJPG')
        out = cv.VideoWriter(output_video_path, fourcc, fps, (width, out_h))

        nasal_resp = [0]
        nostril_visible = []
        plot_buffer = np.zeros(width, dtype=np.uint8)

        # NEW METRICS
        valid_count = 0
        pass_count = 0
        containments = []

        frame_number = 1
        start_time = time.time()

        while cap.isOpened():                         
            ret, frame = cap.read()
            if not ret:
                break

            annotated_frame = frame.copy()

            result = model(frame, verbose=False)[0]
            all_boxes = result.boxes.data.cpu().numpy()

            visible_flag = 0

            if all_boxes.shape[0] != 0:

                face_boxes = all_boxes[all_boxes[:, 5] == FACE_CLASS]
                nose_boxes = all_boxes[all_boxes[:, 5] == NOSTRIL_CLASS]

                # ---- respiration ----
                if len(nose_boxes) > 0:
                    best_nose = nose_boxes[np.argmax(nose_boxes[:, 4])]
                    x1, y1, x2, y2 = map(int, best_nose[:4])

                    x1, y1 = max(0, x1), max(0, y1)
                    x2, y2 = min(width, x2), min(height, y2)

                    roi = frame[y1:y2, x1:x2]

                    if roi.size > 0:
                        nasal_resp.append(np.mean(roi))
                    else:
                        nasal_resp.append(nasal_resp[-1])

                    visible_flag = 1
                else:
                    nasal_resp.append(nasal_resp[-1])

                # ---- containment metric ----
                if len(face_boxes) > 0 and len(nose_boxes) > 0:

                    face = face_boxes[np.argmax(face_boxes[:, 4])][:4]
                    nose = nose_boxes[np.argmax(nose_boxes[:, 4])][:4]

                    face = list(map(int, face))
                    nose = list(map(int, nose))

                    containment = containment_ratio(nose, face)

                    valid_count += 1
                    containments.append(containment)

                    if containment > 0.9:
                        pass_count += 1

                # ---- draw ----
                for box in all_boxes:
                    x1, y1, x2, y2, conf, cls = map(int, box)

                    if cls == FACE_CLASS:
                        color = (0, 255, 0)   # face
                    elif cls == NOSTRIL_CLASS:
                        color = (255, 0, 0)   # nostril
                    else:
                        continue

                    cv.rectangle(annotated_frame, (x1, y1), (x2, y2), color, 1)

            else:
                nasal_resp.append(nasal_resp[-1])

            nostril_visible.append(visible_flag)

            # -------- timeline --------
            plot_buffer[:-1] = plot_buffer[1:]
            plot_buffer[-1] = visible_flag

            canvas = np.zeros((out_h, width, 3), dtype=np.uint8)
            canvas[:height] = annotated_frame

            strip = canvas[height:height + STRIP_HEIGHT]
            strip[:] = (20, 20, 20)

            y_low = STRIP_HEIGHT - 10
            y_high = 10

            ys = np.where(plot_buffer == 1, y_high, y_low)
            xs = np.arange(width)

            pts = np.vstack((xs, ys)).T.reshape(-1, 1, 2)
            cv.polylines(strip, [pts], False, (0, 255, 0), 2)

            cv.line(strip, (0, y_low), (width, y_low), (100, 100, 100), 1)
            cv.line(strip, (width - 1, 0), (width - 1, STRIP_HEIGHT), (255, 255, 255), 1)
            cv.line(canvas, (0, height), (width, height), (255, 255, 255), 1)

            out.write(canvas)
            frame_number += 1

        # -------- SAVE ARRAYS --------
        nasal_resp = np.array(nasal_resp)[1:]
        nostril_visible = np.array(nostril_visible)

        np.save(os.path.join(resp_store_path, vid_names.replace(".avi", ".npy")), nasal_resp)
        np.save(os.path.join(nostril_visible_store_path, vid_names.replace(".avi", ".npy")), nostril_visible)

        cap.release()
        out.release()

        # -------- FINAL METRICS --------
        if valid_count > 0:
            containment_pass_frac = pass_count / valid_count
            mean_containment = np.mean(containments)
        else:
            containment_pass_frac = np.nan
            mean_containment = np.nan

        actual_frames = frame_number - 1                                    
        valid_frac = valid_count / actual_frames if actual_frames > 0 else np.nan  

        # -------- STREAM WRITE --------
        with open(csv_path, "a", newline="") as f:
            writer = csv.writer(f)
            writer.writerow([
                curr_pid,
                vid_names,
                mean_containment,
                containment_pass_frac,
                valid_frac,
                actual_frames
            ])
            f.flush()
            os.fsync(f.fileno())

        print(f"{vid_names} | pass={containment_pass_frac:.3f} | valid={valid_frac:.3f} | time={time.time() - start_time:.2f}s")

## 2.1 Nostril Visibility Summary

Aggregates visibility across all videos per participant. Participants where the nostril was visible in **>50% of frames** are flagged as usable for respiration analysis.

A smoothing window of 50 frames (~2 s at 25 fps) bridges brief occlusion gaps before aggregating.


In [ ]:
def smooth_nostril_visibleness(data, smoothing_window=50):
    """ Smooth the nostril visibleness data using a moving maximum filter. """
    return np.array([np.max(data[max(0, i-smoothing_window):min(len(data), i+smoothing_window)]) for i in range(len(data))])

nasal_visible_df = []

for participant in tqdm(pids):
    total_time = 0
    nasal_invisible = 0
    nasal_dir = os.path.join(data_path,participant,"nostril-visible")
    for nasal_file in np.sort(os.listdir(nasal_dir)):
        nasal = np.load(os.path.join(nasal_dir,nasal_file))
        nasal = smooth_nostril_visibleness(nasal, smoothing_window=50)
        total_time += len(nasal)
        nasal_invisible += np.sum(nasal)
    nasal_visible_df.append({
        "participant": participant,
        "total_time_frames": total_time,
        "nasal_invisible_frames": nasal_invisible,
        "nasal_invisible_percentage": (nasal_invisible/total_time)*100,
        "nasal_visible_percentage": 100-(nasal_invisible/total_time)*100,
        "Use" : (nasal_invisible/total_time)*100 < 50
    })

nasal_visible_df = pd.DataFrame(nasal_visible_df)

print("Participants with less than 50% nasal invisibility:")
print(nasal_visible_df[nasal_visible_df["Use"]==True]["participant"].values)
print("Participant count: ", len(nasal_visible_df[nasal_visible_df["Use"]==True]))

nasal_visible_df

Plot the raw frame-by-frame visibility signal for a single participant to visually inspect occlusion patterns.


In [ ]:
participant = "P01_1"
nasal_dir = os.path.join(data_path,participant,"nostril-visible")
for nasal_file in np.sort(os.listdir(nasal_dir)):
    nasal = np.load(os.path.join(nasal_dir,nasal_file))
    # nasal = smooth_nostril_visibleness(nasal, smoothing_window=50)

    plt.figure(figsize=(20, 5))
    plt.plot(nasal)
    plt.title(f"Nasal Visibleness for {participant} - {nasal_file}")
    plt.ylim(-0.1, 1.1)
    plt.show()

---

# 3. Respiration Signal Extraction

Extracts the actual breathing waveform. For each frame where the nostril is detected, the **mean pixel intensity of the nostril ROI** is recorded as the respiration sample.

**Why mean intensity works:** thermal cameras capture heat emitted from the nostrils — exhaled air is warmer than inhaled air, producing a measurable intensity oscillation at the breathing frequency.

Frames without a detected nostril are stored as `NaN` rather than holding the last value, keeping the signal honest about occlusions.

**Additional outputs vs Section 2:**
```
P01_1/nasal-respf_sig/
├── thermal_rec.npy              # float32 signal array
└── thermal_rec_signal.csv       # time_sec, mean_intensity
```

The annotated video now has two overlay panels at the bottom:
- **Cyan** — live normalised respiration waveform
- **Green** — binary nostril visibility timeline


In [ ]:
import os
import cv2 as cv
import numpy as np
import time
from tqdm import tqdm
import csv

ANNOTATED_DIR_NAME = "annotated_videos_sig"

STRIP_HEIGHT = 160

FACE_CLASS = 0
NOSTRIL_CLASS = 0

csv_path = os.path.join(data_path, "nostril_face_metrics-sig.csv")

# -------- INIT CSV --------
if not os.path.exists(csv_path):
    with open(csv_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            "pid", "video",
            "mean_containment",
            "containment_pass_frac",
            "valid_frac",
            "total_frames"
        ])

# -------- HELPER --------
def containment_ratio(inner, outer):
    xi1, yi1, xi2, yi2 = inner
    xo1, yo1, xo2, yo2 = outer

    inter_x1 = max(xi1, xo1)
    inter_y1 = max(yi1, yo1)
    inter_x2 = min(xi2, xo2)
    inter_y2 = min(yi2, yo2)

    inter_area = max(0, inter_x2 - inter_x1) * max(0, inter_y2 - inter_y1)
    inner_area = (xi2 - xi1) * (yi2 - yi1) + 1e-6

    return inter_area / inner_area


# -------- MAIN LOOP --------
for curr_pid in tqdm(pids, position=0):

    video_store_path = os.path.join(data_path, curr_pid)
    resp_store_path = os.path.join(data_path, curr_pid, "nasal-respf_sig")
    nostril_visible_store_path = os.path.join(data_path, curr_pid, "nostril-visible_sig")
    annotated_store_path = os.path.join(data_path, curr_pid, ANNOTATED_DIR_NAME)

    os.makedirs(resp_store_path, exist_ok=True)
    os.makedirs(nostril_visible_store_path, exist_ok=True)
    os.makedirs(annotated_store_path, exist_ok=True)

    for vid_names in tqdm(
        np.sort(os.listdir(video_store_path)),
        position=1,
        leave=False,
        desc=f"Processing {curr_pid}"
    ):

        if not vid_names.endswith(".avi"):
            continue

        video_to_read = os.path.join(video_store_path, vid_names)
        cap = cv.VideoCapture(video_to_read)

        if not cap.isOpened():
            print(f"Failed to open: {video_to_read}")
            continue

        total_frames = int(cap.get(cv.CAP_PROP_FRAME_COUNT))  # may be wrong for truncated files
        fps = cap.get(cv.CAP_PROP_FPS)
        width = int(cap.get(cv.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv.CAP_PROP_FRAME_HEIGHT))

        out_h = height + STRIP_HEIGHT

        output_video_path = os.path.join(
            annotated_store_path,
            vid_names.replace(".avi", "_annotated.avi")
        )

        fourcc = cv.VideoWriter_fourcc(*'MJPG')
        out = cv.VideoWriter(output_video_path, fourcc, fps, (width, out_h))

        # -------- SIGNAL STORAGE --------
        nasal_resp = []
        nostril_visible = []
        visibility_plot = np.zeros(width, dtype=np.uint8)

        # METRICS
        valid_count = 0
        pass_count = 0
        containments = []

        frame_number = 1
        start_time = time.time()

        while cap.isOpened():                          # ← FIX 1: no total_frames bound

            ret, frame = cap.read()
            if not ret:
                break

            annotated_frame = frame.copy()

            result = model(frame, verbose=False)[0]
            all_boxes = result.boxes.data.cpu().numpy()

            visible_flag = 0
            current_signal = np.nan

            if all_boxes.shape[0] != 0:

                face_boxes = all_boxes[all_boxes[:, 5] == FACE_CLASS]
                nose_boxes = all_boxes[all_boxes[:, 5] == NOSTRIL_CLASS]

                # -------- RESPIRATION SIGNAL --------
                if len(nose_boxes) > 0:

                    best_nose = nose_boxes[np.argmax(nose_boxes[:, 4])]
                    x1, y1, x2, y2 = map(int, best_nose[:4])

                    x1 = max(0, x1)
                    y1 = max(0, y1)
                    x2 = min(width, x2)
                    y2 = min(height, y2)

                    roi = frame[y1:y2, x1:x2]

                    if roi.size > 0:
                        current_signal = np.mean(roi)
                        visible_flag = 1

                # -------- CONTAINMENT METRIC --------
                if len(face_boxes) > 0 and len(nose_boxes) > 0:

                    face = face_boxes[np.argmax(face_boxes[:, 4])][:4]
                    nose = nose_boxes[np.argmax(nose_boxes[:, 4])][:4]

                    face = list(map(int, face))
                    nose = list(map(int, nose))

                    containment = containment_ratio(nose, face)

                    valid_count += 1
                    containments.append(containment)

                    if containment > 0.9:
                        pass_count += 1

                # -------- DRAW BOXES --------
                for box in all_boxes:

                    x1, y1, x2, y2, conf, cls = box
                    x1, y1, x2, y2, cls = map(int, [x1, y1, x2, y2, cls])

                    if cls == FACE_CLASS:
                        color = (0, 255, 0)
                    elif cls == NOSTRIL_CLASS:
                        color = (255, 0, 0)
                    else:
                        continue

                    cv.rectangle(annotated_frame, (x1, y1), (x2, y2), color, 1)

            nasal_resp.append(current_signal)
            nostril_visible.append(visible_flag)

            # ============================================================
            # -------------------- VISIBILITY TIMELINE -------------------
            # ============================================================

            visibility_plot[:-1] = visibility_plot[1:]
            visibility_plot[-1] = visible_flag

            # ============================================================
            # -------------------- CREATE CANVAS -------------------------
            # ============================================================

            canvas = np.zeros((out_h, width, 3), dtype=np.uint8)
            canvas[:height] = annotated_frame

            strip = canvas[height:height + STRIP_HEIGHT]
            strip[:] = (20, 20, 20)

            # ============================================================
            # -------------------- SIGNAL WAVEFORM -----------------------
            # ============================================================

            signal_top = 10
            signal_bottom = 70

            recent_signal = nasal_resp[-width:]
            signal_arr = np.array(recent_signal, dtype=np.float32)
            valid_mask = ~np.isnan(signal_arr)

            if np.sum(valid_mask) > 1:

                valid_vals = signal_arr[valid_mask]
                sig_min = np.min(valid_vals)
                sig_max = np.max(valid_vals)

                if sig_max - sig_min < 1e-6:
                    sig_max = sig_min + 1

                norm_signal = (signal_arr - sig_min) / (sig_max - sig_min)
                ys = signal_bottom - norm_signal * (signal_bottom - signal_top)

                pts = []
                for i in range(len(signal_arr)):
                    if not np.isnan(signal_arr[i]):
                        x = width - len(signal_arr) + i
                        y = int(ys[i])
                        pts.append([x, y])

                if len(pts) > 1:
                    pts = np.array(pts, dtype=np.int32).reshape(-1, 1, 2)
                    cv.polylines(strip, [pts], False, (0, 255, 255), 2)

            cv.putText(strip, "Thermal Respiration Signal", (10, 20),
                       cv.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 1)

            # ============================================================
            # -------------------- VISIBILITY TIMELINE -------------------
            # ============================================================

            vis_top = 95
            vis_bottom = 145

            ys = np.where(visibility_plot == 1, vis_top, vis_bottom)
            xs = np.arange(width)
            pts = np.vstack((xs, ys)).T.reshape(-1, 1, 2)

            cv.polylines(strip, [pts], False, (0, 255, 0), 2)

            cv.putText(strip, "Nostril Visibility", (10, 90),
                       cv.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)

            cv.line(canvas, (0, height), (width, height), (255, 255, 255), 1)

            out.write(canvas)
            frame_number += 1

        # ============================================================
        # -------------------- FINALIZE ARRAYS -----------------------
        # ============================================================

        nasal_resp = np.array(nasal_resp, dtype=np.float32)
        nostril_visible = np.array(nostril_visible, dtype=np.uint8)

        # -------- SAVE NPY --------
        np.save(os.path.join(resp_store_path, vid_names.replace(".avi", ".npy")), nasal_resp)
        np.save(os.path.join(nostril_visible_store_path, vid_names.replace(".avi", ".npy")), nostril_visible)

        # -------- SAVE CSV SIGNAL --------
        time_axis = np.arange(len(nasal_resp)) / fps

        signal_csv_path = os.path.join(resp_store_path, vid_names.replace(".avi", "_signal.csv"))

        with open(signal_csv_path, "w", newline="") as f:
            writer = csv.writer(f)
            writer.writerow(["time_sec", "mean_intensity"])
            for t, val in zip(time_axis, nasal_resp):
                if np.isnan(val):
                    writer.writerow([t, ""])
                else:
                    writer.writerow([t, val])

        cap.release()
        out.release()

        # ============================================================
        # -------------------- FINAL METRICS -------------------------
        # ============================================================

        if valid_count > 0:
            containment_pass_frac = pass_count / valid_count
            mean_containment = np.mean(containments)
        else:
            containment_pass_frac = np.nan
            mean_containment = np.nan

        actual_frames = frame_number - 1                                           # ← FIX 2
        valid_frac = valid_count / actual_frames if actual_frames > 0 else np.nan  # ← FIX 3

        # -------- STREAM WRITE --------
        with open(csv_path, "a", newline="") as f:
            writer = csv.writer(f)
            writer.writerow([
                curr_pid,
                vid_names,
                mean_containment,
                containment_pass_frac,
                valid_frac,
                actual_frames
            ])
            f.flush()
            os.fsync(f.fileno())

        print(
            f"{vid_names} | "
            f"pass={containment_pass_frac:.3f} | "
            f"valid={valid_frac:.3f} | "
            f"time={time.time() - start_time:.2f}s"
        )